# Foraward simulation for sandbox

In [1]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## test

In [ ]:
import pandas as pd


class SLE:

    def __init__(self):

        # 井資料庫
        self.wells = []

        # 範例 node table
        self.node = pd.DataFrame({
            "node_id": [13, 14, 8, 10],
            "x": [100, 200, 300, 250],
            "y": [50, 50, 100, 80]
        })


    def select_nodes(self, by, value):

        if by.lower() != "coordinate":
            raise ValueError("Only coordinate selection is implemented.")

        x, y = value

        return self.node[
            (self.node["x"] == x) &
            (self.node["y"] == y)
        ]


    def add_wells(self, *wells):
        """
        Add wells.

        Parameters
        ----------
        wells :
            (
                stress,
                coordinate,
                name,
                type,
                parameter(dict)
            )
        """

        for well in wells:

            stress, coordinate, name, well_type, parameter = well

            df_node = self.select_nodes(
                by="coordinate",
                value=coordinate
            )

            if df_node.empty:
                raise ValueError(
                    f"Cannot find node at {coordinate}"
                )

            node_id = int(df_node.iloc[0]["node_id"])

            self.wells.append({
                "stress": stress,
                "node_id": node_id,
                "coordinate": coordinate,
                "name": name,
                "type": well_type.lower(),
                "parameter": parameter
            })

        return self

    
    def create_observation(self):

        rows = []

        # by "observation" 
        stress_list = sorted(
            {w["stress"] for w in self.wells
             if w["type"] == "observation"}
        )
        
        for stress_idx in stress_list:
            
            # by "stress"
            obs = [
                w for w in self.wells
                if w["stress"] == stress_idx 
                and w["type"] == "observation"
            ]

            rows.append([stress_idx, len(obs)])

            for well in obs:

                rows.append([ well["node_id"], well["name"]])

        return pd.DataFrame(rows)


    def create_source(self):


        # select by well type
        stress_list = sorted(
            {w["stress"] for w in self.wells
             if w["type"] == "injection"}
        )
        
        df_up = pd.DataFrame([[len(stress_list)], [0]]) # for [total stress number] and [0]
        
        src_list = [df_up]
        # store each stress
        for stress_idx in stress_list:

            src = [
                w for w in self.wells
                if w["stress"] == stress_idx and  w["type"] == "injection"
            ]

            rows = []
            df_between = pd.DataFrame([len(src)])

            for well in src:

                rows.append([
                    well["node_id"],              # node idex
                    well["parameter"]["rate"],    # sink/source for flow
                    0,                            # sink/source for concentration
                    well["parameter"]["time"][0], # start time
                    well["parameter"]["time"][1], # end time
                    0,                            # start time for concentration
                    0,                            # end time for concentration
                    well["name"],                 # well name
                ])
                
                df = pd.DataFrame(rows)
            
            src_list.append(pd.concat([df_between, df]))
            
        
        return pd.concat(src_list)

 
    def create_times(self, *times):
        """ 
        Add time.

        Parameters
        ----------
        t1 =  (dt, dt_max, dt_multipler, t_max, output_flag, t_reduction)
        
        """
        t_list = []
        for t_ in times:
            dt, dt_max, dt_m, t_max, flag, t_re = t_
            
            if flag == 'time':
                flag_value = 0
            
            t_list.append([
                dt, 
                dt_max, 
                dt_m,
                t_max,
                flag_value,
                t_re
            ])
        
        
        return pd.DataFrame(t_list).T   
            
        
        
    


In [3]:
model = SLE()

In [6]:
ob1 = ( 1, (100, 50), "OBS-1", "observation", {} )
ob2 = ( 1, (100, 50), "OBS-2", "observation", {} )
ob3 = ( 2, (100, 50), "OBS-3", "observation", {} )
inj1 = ( 1, (100, 50), "INJ_1", "injection", {'rate': 10, 'time': (0, 20)} )
inj2 = ( 1, (100, 50), "INJ_2", "injection", {'rate': 10, 'time': (0, 20)} )
inj3 = ( 3, (100, 50), "INJ_3", "injection", {'rate': 10, 'time': (0, 20)} )

model.add_wells(
    ob1,
    ob2,
    ob3,
    inj1,
    inj2,
    inj3,
    
)

# print(model.wells)
# print(model.create_observation())
# print(model.create_source())


t_1 = (5, 10, 1, 600, 'time', 0)      
t_2 = (10, 10, 1, 600, 'time', 0)
t_3 = (15, 10, 1, 600, 'time', 0)    

print(model.create_times(t_1, t_2, t_3))



     0    1    2
0    5   10   15
1   10   10   10
2    1    1    1
3  600  600  600
4    0    0    0
5    0    0    0


## Simulation control (User input)

In [3]:
# simulation control

# 把一些事先要定義的東西寫好
# STRESS NUMBER?
# CREATE OUTPUT?
# FLOW TYPE?
# INITIAL HEAD TYPE? 
project_name = 'test_0701'
simulation_control = ('3D', 'steady', 'confined')  # dimension, problem, aquifer

# Geometry control
dx = np.array( [43.5, 25, 20, 20, 20] + [7.5] * 19 + [6, 6, 5.5, 5.5, 6, 6] + [7.5] * 19 + [20, 20, 20, 25, 43.5])
dy = np.array( [31, 20, 20, 20, 10] + [7.5] * 24 + [10, 20, 20, 20, 31])
dz = np.array( [30, 25, 20, 15, 10, 10] + [5] * 11 + [7.5, 7.5] + [10, 10, 10, 25, 30, 35] )

start_coord = (0, 0, 0)         # origin coordinate (x0, y0, z0)
element_num = (54, 34, 25)      # element number in each direction (e_x, e_y, e_z)
element_spacing = (dx, dy, dz)  # element spacing in each direction (len(dx) = e_x-)

# initial condition
init_paras = (1.71, 0, 0.0001, 0.00001, 0.4) # init_h, init_flux, init_K, init_Ss, porosity

# boundary condition
boundary = ('surface', 'UP')  # method, value

## 之後換成用迴圈產生這些dict
# wells (stress_idx, coordinate, well_name, role, *{rate: ..., time: ...})

# sink/sources well
const_rate = 600
inj_time = (0, 600)
inj_1 = (1, (288.5, 71,  120), 'inj_1', 'injection', {'rate':const_rate, 'time': inj_time})
inj_2 = (1, (88.5,  71,  110), 'inj_2', 'injection', {'rate':const_rate, 'time': inj_time})
inj_3 = (1, (88.5,  191, 100), 'inj_3', 'injection', {'rate':const_rate, 'time': inj_time})
inj_4 = (1, (88.5,  311, 90 ), 'inj_4', 'injection', {'rate':const_rate, 'time': inj_time})
inj_5 = (1, (288.5, 311, 120), 'inj_5', 'injection', {'rate':const_rate, 'time': inj_time})
inj_6 = (1, (488.5, 311, 110), 'inj_6', 'injection', {'rate':const_rate, 'time': inj_time})
inj_7 = (1, (488.5, 191, 100), 'inj_7', 'injection', {'rate':const_rate, 'time': inj_time})
inj_8 = (1, (488.5, 71,  90 ), 'inj_8', 'injection', {'rate':const_rate, 'time': inj_time})

# observation well
ob_1 = (1, (188.5, 71,  130), 'Pie_1', 'observation', {})
ob_2 = (1, (88.5,  131, 200), 'Pie_2', 'observation', {})
ob_3 = (1, (88.5,  251, 180), 'Pie_3', 'observation', ())
ob_4 = (1, (188.5, 311, 150), 'Pie_4', 'observation', ())
ob_5 = (1, (388.5, 311, 150), 'Pie_5', 'observation', ())
ob_6 = (1, (488.5, 251, 180), 'Pie_6', 'observation', ())
ob_7 = (1, (488.5, 131, 200), 'Pie_7', 'observation', ())
ob_8 = (1, (388.5, 71,  130), 'Pie_8', 'observation', ())


# time info for transient (dt, dt_max, dt_mul, t_max, _, _)
t_1 = (10, 10, 1, 600, 2, 0)      
t_2 = (10, 10, 1, 600, 2, 0)
t_3 = (10, 10, 1, 600, 2, 0)      
t_4 = (10, 10, 1, 600, 2, 0)      
t_5 = (10, 10, 1, 600, 2, 0)      
t_6 = (10, 10, 1, 600, 2, 0)      
t_7 = (10, 10, 1, 600, 2, 0)      
t_8 = (10, 10, 1, 600, 2, 0)                                           
                                        

## Import sle_io package

In [4]:
from sleio import *

wirting_dir = '\forward'
test_forward = sle_io(project_name)
test_forward.set_parameters(simulation_control)
test_forward.create_geometry(start_coord, element_num, element_spacing)
test_forward.create_initial(init_paras)
test_forward.create_boundary(boundary)
test_forward.create_times(t_1, t_2, t_3, t_5, t_6, t_7, t_8)
test_forward.create_source(inj_1, inj_2, inj_3, inj_4, inj_5, inj_6, inj_7, inj_8,)
test_forward.create_observation(ob_1, ob_2, ob_3, ob_4, ob_5, ob_6, ob_7, ob_8)
test_forward.create_function()
test_forward.create_simulation()
test_forward._create_problem()

Project: test_0701
Simulation with '3D' case, 'confined' aquifer under 'steady' state


ValueError: If using all scalar values, you must pass an index

In [ ]:
# simulation control
project_name = 'test_0701'
simulation_control = ('3D', 'confined', 'steady')  # dimension, aquifer_type, simulation state

# Geometry info
start_coord = (0, 0, 0)
element_list = (54, 34, 25) # ele_num_x, ele_num_y, ele_num_z  ! 建立防呆，多少個element提供多少個dx 需要的欄位

dx = np.array( [43.5, 25, 20, 20, 20] + [7.5] * 19 + [6, 6, 5.5, 5.5, 6, 6] + [7.5] * 19 + [20, 20, 20, 25, 43.5])
dy = np.array( [31, 20, 20, 20, 10] + [7.5] * 24 + [10, 20, 20, 20, 31])
dz = np.array( [30, 25, 20, 15, 10, 10] + [5] * 12 + [7.5, 7.5] + [10, 10, 10, 25, 30, 35] )
element_spacing = (dx, dy, dz)

# initial data
init_para = (1.71, 0.0001, 0.00001, 0.4) # init_h, init_K, init_Ss, porosity

# aquifer type
aquifer_type = 0 # confined

# boundary  ! 建立可以包含"點" 以及"面"的邊界條件寫入方法
boundary_node = [

    [2    , 0.875, init_h],
    [1.875, 0.75 , init_h],
    [2.125, 0.75,  init_h],
    [1.875, 0.625, init_h],
    [2    , 0.625, init_h],
    [2.125, 0.625, init_h],  
    [1.75, 0.625,  init_h],  
    [2    , 0.5,   init_h],
]



# obwell ! 建立防呆，一樣多少個obs_num要輸入多少的觀測井資訊
w1 = [922,  'w1']
w2 = [3308, 'w2']
w3 = [4506, 'w3']
w4 = [3410, 'w4']
w5 = [2234, 'w5']
w6 = [4702, 'w6']
w7 = [2336, 'w7']
w8 = [1158, 'w8']
wm = [3416, 'wm']

# sources
t_start, t_end = [0, 600] 
well_name = ['w1', 'w2', 'w3', 'w4', 'w5', 'w6', 'w7', 'w8', 'wm'] 
inj_intensity = np.array([
                        1.55, # w1
                        1.50, ## w2
                        1.50, ## w3
                        1.49, ## w4
                        1.54, ## w5
                        1.33, ## w6
                        1.39, ## w7
                        1.44, ## w8  
                        1.54  ## wm
                        ])


# create forward files  ! 此處將搭配sleio.write_forward_input() 並劃出網格圖
for i , well in enumerate(well_name): 
    os.chdir(os.path.join(work_dir, proj_name, 'forward'))
    try:
        os.mkdir(well)  
    except:
        pass

    os.chdir(well) 
    src_1 =  [pump_node[i], inj_intensity[i], t_start, t_end, well]
    src_list = [ 
        [src_1] 
    ]
        
    sle_forward = sleio(f'3d_yongjhen_demo_{well}_injection')
    sle_forward.grid_element(element_list)
    sle_forward.node(start_coord = start_coord , element_spacing=[dx, dy, dz], init_h = init_h)
    sle_forward.material(init_K, init_Ss, porosity = porosity)
    sle_forward.sources(src_list)
    sle_forward.boundary_node(boundary_node, 1, init_h)
    sle_forward.obwell([ob_list[i]])
    sle_forward.function(aquifer_type)
    time_list = [[10, 10, 1, 600, 2, 0]] # dt, dt_max, dt_mul, t_max, ...
    prob_list = [1, 1, 1, 1, 1]            # forward(1)/inverse(0), create output.dat, transient(2)/steady(1), ...
    simu_list = [2, 2, 1, 50, 0.5, 0.5, 0.001, 10e-10, 10e-10, 10e-10]
    sle_forward.time(time_list)
    sle_forward.simulation(prob_list, simu_list, plot_domain=True)
    sle_forward.write_forward_input(os.getcwd())


## test forward simulation

In [ ]:
#%% test forward simulation (try previous sle result)
# os.chdir(os.path.join(proj_dir, 'inverse_after_clogging'))
# df_result_lnK_mean, df_result_lnK_var = parse_estimation('O-kestimate.dat', dim =3)
os.chdir(os.path.join(proj_dir, 'inverse'))
df_result = pd.read_csv('HTNN_result.csv')

test_list = []
for i , well in enumerate(well_name): 
    os.chdir(os.path.join(work_dir, proj_name, 'forward', well))
    sle_forward_htnn = sleio(f'test_forward with {well} injection')
    k_est =  np.exp(df_result['HTNN_after_clogging'].values)
    df = sle_forward_htnn.run_forward( k_est, 0.00025, 'steady', init_h , plot_head = False)
    test_list.append(df.values[0])
simulation_head = np.concatenate(test_list, axis = 0)
# print(simulation_head.shape)

### Create multiple forward 

In [ ]:
# write forward files
inj_ws = [
    'inj_1','inj_2','inj_3','inj_4','inj_5','inj_6','inj_7','inj_8',
]

for i in range(len(inj_ws)):

    wirting_dir = f'\forward {inj_ws[i]}'
    if 
        os.mkdir()
    
    else:
    
    test_forward = sle_io(project_name)
    test_forward.set_parameters(simulation_control)
    test_forward.add_geometry(start_coord, element_num, element_spacing)
    test_forward.add_initial(init_paras)